In [1]:
import dac
from audiotools import AudioSignal

# Download a model
model_path = dac.utils.download(model_type="44khz")
model = dac.DAC.load(model_path)
model.to('cpu')

# Load audio signal file
signal = AudioSignal('reference1.wav')
signal.to_mono().resample(44100)
print(f"{model.sample_rate=} {signal.sample_rate=}")

# Encode audio signal as one long file
# (may run out of GPU memory on long files)
signal.to(model.device)

x = model.preprocess(signal.audio_data, signal.sample_rate)
z, codes, latents, _, _ = model.encode(x)

# Decode audio signal
y = model.decode(z)


2025-03-26 09:45:51.066559: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-26 09:45:51.081517: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743007551.098956  306651 cuda_dnn.cc:8312] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743007551.104153  306651 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-26 09:45:51.122555: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

model.sample_rate=44100 signal.sample_rate=44100


In [5]:
657920/1285

512.0

In [10]:

# Alternatively, use the `compress` and `decompress` functions
# to compress long files.

signal = signal.cpu()
x = model.compress(signal)

# Save and load to and from disk
x.save("compressed.dac")
x = dac.DACFile.load("compressed.dac")

# Decompress it back to an AudioSignal
y = model.decompress(x)

# Write to file
y.write('output44k.wav')

In [17]:
from audiotools import AudioSignal
from IPython.display import Audio
fname = ['reference1.wav', 'output16k.wav', 'output44k.wav'] 
signal = AudioSignal(fname[2])
# signal.to_mono().resample(44100)
Audio(signal.audio_data.squeeze(), rate=signal.sample_rate, autoplay=True)
